In [1]:
from pathlib import Path
import glob
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tensorboard.backend.event_processing import event_accumulator

ROOT = Path.cwd().parent.parent
LOG_ROOT = ROOT / "scripts" / "rsl_rl" / "logs" / "rsl_rl"

print("Workspace root:", ROOT)
print("Log root:", LOG_ROOT)


Workspace root: /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control
Log root: /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control/scripts/rsl_rl/logs/rsl_rl


## Out-of-distribution generalization

Train both policies on 1D goal sampling, then evaluate the same checkpoints on 3D random goals.
The `play.py` script writes evaluation trajectory JSON next to the checkpoint directory, so the notebook can compare the train curve and the OOD test rollouts from one run root.

Example commands:

```bash
python train.py --task Quadcontrol-1DGoal-v0 --headless --video --seed 0 --experiment_name ood_generalization/vanilla_1d --max_iterations 300
python train.py --task Quadcontrol-1DGoal-SymmAugMirror-v0 --headless --video --seed 0 --experiment_name ood_generalization/symm_mirror_1d --max_iterations 300
python play.py --task Quadcontrol-3DGoal-v0 --checkpoint /path/to/checkpoint.pt --headless
```


In [2]:
EXPERIMENTS = {
    "Vanilla PPO": {
        "run_root": LOG_ROOT / "ood_generalization" / "vanilla_1d",
        "color": "#1f77b4",
    },
    "SymmAug + Mirror Loss": {
        "run_root": LOG_ROOT / "ood_generalization" / "symm_mirror_1d",
        "color": "#d62728",
    },
}

for label, spec in EXPERIMENTS.items():
    print(label)
    print("  run root:", spec["run_root"])


Vanilla PPO
  run root: /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control/scripts/rsl_rl/logs/rsl_rl/ood_generalization/vanilla_1d
SymmAug + Mirror Loss
  run root: /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control/scripts/rsl_rl/logs/rsl_rl/ood_generalization/symm_mirror_1d


In [3]:
def load_tensorboard_scalars(run_root: Path, tag: str = "Train/mean_reward") -> pd.DataFrame:
    rows = []
    event_files = sorted(run_root.glob("**/events.out.tfevents*"))
    for event_file in event_files:
        reader = event_accumulator.EventAccumulator(str(event_file.parent))
        reader.Reload()
        scalar_tags = reader.Tags().get("scalars", [])
        if tag not in scalar_tags:
            continue
        run_name = event_file.parent.name
        for scalar in reader.Scalars(tag):
            rows.append(
                {
                    "step": scalar.step,
                    "value": scalar.value,
                    "tag": tag,
                    "run": run_name,
                    "source": str(event_file.parent),
                }
            )
    return pd.DataFrame(rows)


def load_eval_trajectory_metrics(run_root: Path, success_threshold: float = 0.25) -> pd.DataFrame:
    rows = []
    json_files = sorted(run_root.glob("**/eval_logs/*.json"))
    for json_file in json_files:
        with json_file.open() as handle:
            trajectory = pd.DataFrame(json.load(handle))
        if trajectory.empty:
            continue
        pos = trajectory[["x", "y", "z"]].to_numpy()
        ref = trajectory[["xref", "yref", "zref"]].to_numpy()
        distance = np.linalg.norm(pos - ref, axis=1)
        final_distance = float(distance[-1])
        mean_distance = float(distance.mean())
        rows.append(
            {
                "episode": json_file.stem,
                "run": json_file.parent.parent.name,
                "final_distance": final_distance,
                "mean_distance": mean_distance,
                "success": float(final_distance < success_threshold),
                "source": str(json_file),
            }
        )
    return pd.DataFrame(rows)


In [4]:
train_frames = []
test_frames = []
summary_rows = []

for label, spec in EXPERIMENTS.items():
    run_root = spec["run_root"]

    train_df = load_tensorboard_scalars(run_root)
    if not train_df.empty:
        train_df = train_df.assign(label=label)
        train_frames.append(train_df)
        final_train = train_df.sort_values("step").groupby("run").tail(1)
        summary_rows.append(
            {
                "label": label,
                "split": "train",
                "metric": "final_reward",
                "mean": final_train["value"].mean(),
                "std": final_train["value"].std(ddof=0),
                "n": final_train["run"].nunique(),
            }
        )
    else:
        print(f"No training scalars found under {run_root}")

    test_df = load_eval_trajectory_metrics(run_root)
    if not test_df.empty:
        test_df = test_df.assign(label=label)
        test_frames.append(test_df)
        summary_rows.append(
            {
                "label": label,
                "split": "test",
                "metric": "final_distance",
                "mean": test_df["final_distance"].mean(),
                "std": test_df["final_distance"].std(ddof=0),
                "n": len(test_df),
            }
        )
    else:
        print(f"No evaluation trajectories found under {run_root}")

train_df = pd.concat(train_frames, ignore_index=True) if train_frames else pd.DataFrame()
test_df = pd.concat(test_frames, ignore_index=True) if test_frames else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows)
summary_df


No training scalars found under /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control/scripts/rsl_rl/logs/rsl_rl/ood_generalization/vanilla_1d
No evaluation trajectories found under /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control/scripts/rsl_rl/logs/rsl_rl/ood_generalization/vanilla_1d
No training scalars found under /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control/scripts/rsl_rl/logs/rsl_rl/ood_generalization/symm_mirror_1d
No evaluation trajectories found under /home/vikas-admin/Codebases/SPEAR/MetaRL/ss26-drone-control/scripts/rsl_rl/logs/rsl_rl/ood_generalization/symm_mirror_1d


""


In [5]:
sns.set_style("whitegrid")
plt.style.use(["science", "ieee"])

if train_df.empty and test_df.empty:
    print("Add training and evaluation logs under the directories above, then rerun this notebook.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

    if not train_df.empty:
        sns.lineplot(
            data=train_df,
            x="step",
            y="value",
            hue="label",
            estimator="mean",
            errorbar="sd",
            palette={label: spec["color"] for label, spec in EXPERIMENTS.items()},
            ax=axes[0],
        )
        axes[0].set_title("Training on 1D goals")
        axes[0].set_xlabel("Environment steps")
        axes[0].set_ylabel("Train/mean_reward")
        axes[0].legend(title="Policy")

    if not test_df.empty:
        eval_summary = (
            test_df.groupby("label", as_index=False)
            .agg(
                final_distance_mean=("final_distance", "mean"),
                final_distance_std=("final_distance", "std"),
                success_rate=("success", "mean"),
                episodes=("episode", "count"),
            )
        )
        order = [label for label in EXPERIMENTS.keys() if label in eval_summary["label"].tolist()]
        eval_summary = eval_summary.set_index("label").loc[order].reset_index()
        sns.barplot(
            data=eval_summary,
            x="label",
            y="final_distance_mean",
            order=order,
            palette={label: spec["color"] for label, spec in EXPERIMENTS.items()},
            ax=axes[1],
            errorbar=None,
        )
        axes[1].errorbar(
            x=np.arange(len(eval_summary)),
            y=eval_summary["final_distance_mean"],
            yerr=eval_summary["final_distance_std"].fillna(0.0),
            fmt="none",
            ecolor="black",
            capsize=4,
            linewidth=1,
        )
        axes[1].set_title("OOD test on 3D goals")
        axes[1].set_xlabel("")
        axes[1].set_ylabel("Final distance to goal")
        axes[1].tick_params(axis="x", rotation=15)
        for idx, row in eval_summary.reset_index(drop=True).iterrows():
            axes[1].text(idx, row["final_distance_mean"] + 0.02, f"succ={row['success_rate']:.2f}", ha="center", va="bottom")

    plt.show()


OSError: 'science' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)

## Rliable

In [6]:
from rliable import library as rly
from rliable import metrics
from rliable import plot_utils
import numpy as np

#@title Plotting: Seaborn style and matplotlib params
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import MaxNLocator
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style("white")

# Matplotlib params
from matplotlib import rcParams
from matplotlib import rc

rcParams['legend.loc'] = 'best'
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42

rc('text', usetex=False)

In [7]:
# create reward dataframe to be used with rliable
summary.head()

# aggreate reward over the last 50 steps for each run
final_rewards = summary.groupby(["task", "run"]).apply(lambda x: x.sort_values("step").tail(50)["value"].mean()).reset_index(name="final_reward")
print(final_rewards)
# drop run column
final_rewards = final_rewards.drop(columns=["run"])
print(final_rewards)

                     task                                  run  final_reward
0   mirror_loss_only/0p05  2026-04-24_16-34-04_mlc_0p05_seed_0    118.974223
1   mirror_loss_only/0p05  2026-04-24_16-36-16_mlc_0p05_seed_1    129.187053
2   mirror_loss_only/0p05  2026-04-24_16-38-25_mlc_0p05_seed_2    125.435778
3   mirror_loss_only/0p20  2026-04-24_16-40-37_mlc_0p20_seed_0    126.454014
4   mirror_loss_only/0p20  2026-04-24_16-42-48_mlc_0p20_seed_1    127.264583
5   mirror_loss_only/0p20  2026-04-24_16-44-59_mlc_0p20_seed_2    125.847228
6   mirror_loss_only/1p00  2026-04-24_16-47-08_mlc_1p00_seed_0    124.889284
7   mirror_loss_only/1p00  2026-04-24_16-49-20_mlc_1p00_seed_1    129.819786
8   mirror_loss_only/1p00  2026-04-24_16-51-27_mlc_1p00_seed_2    118.612052
9   mirror_loss_only/2p00  2026-04-24_16-53-37_mlc_2p00_seed_0    125.405964
10  mirror_loss_only/2p00  2026-04-24_16-55-50_mlc_2p00_seed_1    119.609936
11  mirror_loss_only/2p00  2026-04-24_16-58-00_mlc_2p00_seed_2    121.321281

/tmp/ipykernel_145029/2624729464.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rewards = summary.groupby(["task", "run"]).apply(lambda x: x.sort_values("step").tail(50)["value"].mean()).reset_index(name="final_reward")


In [8]:
# make a dataframe with the final rewards for each task and run , don't need run as a column
final_rewards

# nomralise the final rewards by the min and max reward achieved across all runs and tasks
min_reward = final_rewards["final_reward"].min()
max_reward = final_rewards["final_reward"].max()
final_rewards["normalized_final_reward"] = (final_rewards["final_reward"] - min_reward) / (max_reward - min_reward)
print(final_rewards)

                     task  final_reward  normalized_final_reward
0   mirror_loss_only/0p05    118.974223                 0.032314
1   mirror_loss_only/0p05    129.187053                 0.943545
2   mirror_loss_only/0p05    125.435778                 0.608841
3   mirror_loss_only/0p20    126.454014                 0.699692
4   mirror_loss_only/0p20    127.264583                 0.772014
5   mirror_loss_only/0p20    125.847228                 0.645552
6   mirror_loss_only/1p00    124.889284                 0.560080
7   mirror_loss_only/1p00    129.819786                 1.000000
8   mirror_loss_only/1p00    118.612052                 0.000000
9   mirror_loss_only/2p00    125.405964                 0.606181
10  mirror_loss_only/2p00    119.609936                 0.089035
11  mirror_loss_only/2p00    121.321281                 0.241729


In [9]:
#@title Stratified Bootstrap CIs and Aggregate metrics

StratifiedBootstrap = rly.StratifiedBootstrap

IQM = lambda x: metrics.aggregate_iqm(x) # Interquartile Mean
OG = lambda x: metrics.aggregate_optimality_gap(x, 1.0) # Optimality Gap
MEAN = lambda x: metrics.aggregate_mean(x)
MEDIAN = lambda x: metrics.aggregate_median(x)

In [10]:
#@title Score helpers -- Subsampling


def subsample_scores(score_dict, n=5, replace=False):
  subsampled_dict = {}
  total_samples = len(score_dict[list(score_dict.keys())[0]])
  for game, scores in score_dict.items():
    indices = np.random.choice(range(total_samples), size=n, replace=replace)
    subsampled_dict[game] = scores[indices]
  return subsampled_dict

def subsample_scores_mat(score_mat, num_samples=5, replace=False):
  subsampled_dict = []
  total_samples, num_games = score_mat.shape
  subsampled_scores = np.empty((num_samples, num_games))
  for i in range(num_games):
    indices = np.random.choice(total_samples, size=num_samples, replace=replace)
    subsampled_scores[:, i] = score_mat[indices, i]
  return subsampled_scores

def subsample_seeds(score_mat, num_samples=5, replace=False):
  indices = np.random.choice(
      score_mat.shape[0], size=num_samples, replace=replace)
  return score_mat[indices]

def batch_subsample_seeds(score_mat, num_samples=5, batch_size=100,
                          replace=False):
  indices = [
    np.random.choice(score_mat.shape[0], size=num_samples, replace=replace)
    for _ in range(batch_size)
  ]
  return (score_mat[idx] for idx in indices)

def subsample_scores_mat_with_replacement(score_mat, num_samples=5):
  subsampled_dict = []
  total_samples, num_games = score_mat.shape
  indices = np.random.choice(
      total_samples, size=(num_samples, num_games), replace=True)
  col_indices =  np.expand_dims(np.arange(num_games), axis=0)
  col_indices = np.repeat(col_indices, num_samples, axis=0)
  subsampled_scores = score_mat[indices, col_indices]
  return subsampled_scores

In [11]:
#@title Aggregate computation helpers

SIZES = [3, 5, 10, 25, 50, 100]

def calc_aggregate_fn(score_data, num_samples=5, total_n=20000, 
                      aggregate_fn=MEDIAN, replace=False):
  subsampled_scores = batch_subsample_seeds(
      score_data, num_samples, batch_size=total_n, replace=replace)
  aggregates = [aggregate_fn(scores) for scores in subsampled_scores]
  return np.array(aggregates)

def calculate_aggregate_varying_sizes(score_matrix, aggregate_fn, total_n=20000,
                                      sizes=None, replace=False):
  agg_dict = {}
  if sizes is None:
    sizes = SIZES
  for size in sizes:
    agg_dict[n] = calc_aggregate_fn(score_matrix, num_samples=size, aggregate_fn=aggregate_fn,
                                    total_n=total_n, replace=replace)
    print('Mean Aggregate: {}'.format(np.mean(agg_dict[n])))
  return agg_dict

def CI(bootstrap_dist, stat_val=None, alpha=0.05, is_pivotal=False):
    """
    Get the bootstrap confidence interval for a given distribution.
    Args:
      bootstrap_distribution: numpy array of bootstrap results.
      stat_val: The overall statistic that this method is attempting to
        calculate error bars for. Default is None.
      alpha: The alpha value for the confidence intervals.
      is_pivotal: if true, use the pivotal (reverse percentile) method. 
        If false, use the percentile method.
    Returns:
      (low, high): The lower and upper limit for `alpha` x 100% CIs.
      val: The median value of the bootstrap distribution if `stat_val` is None
        else `stat_val`.
    """
    # Adapted from https://pypi.org/project/bootstrapped
    if is_pivotal:
      assert stat_val is not None, 'Please pass the statistic for a pivotal'
      'confidence interval' 
      low = 2 * stat_val - np.percentile(bootstrap_dist, 100 * (1 - alpha / 2.))
      val = stat_val
      high = 2 * stat_val - np.percentile(bootstrap_dist, 100 * (alpha / 2.))
    else:
      low = np.percentile(bootstrap_dist, 100 * (alpha / 2.))
      val = np.percentile(bootstrap_dist, 50)
      high = np.percentile(bootstrap_dist, 100 * (1 - alpha / 2.))
    return (low, high), val

In [12]:
#@title Aggregates on Atari 100K (with 10 runs)


aggregate_func = lambda x: np.array([MEDIAN(x), IQM(x), MEAN(x), OG(x)])
aggregate_scores, aggregate_interval_estimates = rly.get_interval_estimates(
    atari_100k_score_dict, aggregate_func, reps=50000)

algorithms = ['SimPLe', 'DER', 'OTR', 'CURL', 'DrQ', 'DrQ(ε)', 'SPR']
fig, axes = plot_utils.plot_interval_estimates(
    aggregate_scores, 
    aggregate_interval_estimates,
    metric_names = ['Median', 'IQM', 'Mean', 'Optimality Gap'],
    algorithms=algorithms,
    colors=ATARI_100K_COLOR_DICT,
    xlabel_y_coordinate=-0.16,
    xlabel='Human Normalized Score')
plt.show()
save_fig(fig, 'atari_100k_aggregates')

NameError: name 'atari_100k_score_dict' is not defined

## Reward distribution for each task